# Lab 00 · From rules to LLM calls

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## One set of questions, two ways to answer
**Start with a problem:** A rule can only use facts and conditions someone has written. What happens when the same architecture question is asked in natural language?

**In this lab:** Apply a retry or pattern rule, then ask an LLM the same questions. Compare both answers and inspect disagreements.

**Goal:** Compare rules with an LLM while applying software architecture concepts: retries, idempotency, bulkheads, API gateways, and BFFs.

ShopEasy's services communicate over a network. A call may fail temporarily, one slow dependency may consume shared resources, and different clients may need different APIs.

First, answer short multiple-choice questions using explicit Python rules. Then make one simple LLM call before asking the model the **same questions**, without giving it those rules, and comparing its answers. The model is not an agent yet: it makes one independent call per question and uses no tools.

These are simplified teaching rules. Real designs require more context, and automatic retries need timeouts, a limit, and backoff with jitter.

**At a glance**

![Lab 00 at a glance: 7 quiz questions (retries + service patterns), then Apply the rule (rule_answer()) or Ask a model (ask() for one letter), then Compare (agreement count).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/00.svg)

## Think about the example

A retry repeats a failed request; a timeout can leave the result uncertain. Bulkheads separate resources, an API gateway handles shared edge concerns, and a BFF adapts an API to a client.

**Concepts:** Safe retries and idempotency; Bulkheads; API gateways and BFFs.

Use the scenario above to consider these questions before running the code:

1. When is a retry safe, and why is a timed-out payment uncertain?
2. How do bulkheads, API gateways, and BFFs address different problems?

## The idea in context

A Python rule checks explicit facts; an LLM reads a natural-language question and returns a response. Both answers need to be reviewed in context.

Retries need a transient failure, a safe operation, a limit, and backoff. Bulkheads isolate resources; gateways and BFFs serve different edge needs.

| Key term | Meaning |
| --- | --- |
| **Idempotency** | Repeating an operation has the same intended effect as performing it once. |
| **Bulkhead** | A resource boundary that keeps one dependency from exhausting another’s capacity. |
| **API gateway** | An entry point that can route external requests and apply shared policies. |
| **Backend for frontend (BFF)** | A backend that adapts an API to a particular client’s needs. |
| **LLM prompt** | The instructions, facts, and question sent to a language model. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For live mode, save `DEEPSEEK_API_KEY` in Colab Secrets, keep `PROVIDER="deepseek"`, set `MODE="live"`, and rerun setup. You can also choose `"gemini"` or `"freeinference"` with the corresponding secret. `ask` stops after 12 calls; SDK and framework examples make separate calls.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Read the questions
Each case contains its question, answer choices, and facts for the rule. `idempotent=True` means repeating the request has the same intended effect as one request. For a payment, this requires a server-supported idempotency key reused on every attempt; a key alone is not enough.

Read the questions and choose your answers before running the rules. Retry questions assume a transient failure unless stated otherwise. A timeout does **not** prove that a payment failed.

In [ ]:
retry_choices = ["A. Retry with a limit and backoff", "B. Do not retry automatically"]
pattern_choices = ["A. Bulkhead", "B. API gateway", "C. BFF"]
cases = [
    {"name": "Product read", "kind": "retry", "transient": True, "idempotent": True,
     "question": "A read-only product lookup times out due to a temporary network failure. Retry?", "choices": retry_choices},
    {"name": "Unsafe payment", "kind": "retry", "transient": True, "idempotent": False,
     "question": "A temporary network failure causes a card charge to time out. Repeating it may charge twice. Retry?", "choices": retry_choices},
    {"name": "Safe payment", "kind": "retry", "transient": True, "idempotent": True,
     "question": "A charge times out due to a temporary network failure. The server guarantees deduplication using the same idempotency key. Retry?", "choices": retry_choices},
    {"name": "Invalid request", "kind": "retry", "transient": False, "idempotent": True,
     "question": "A product read gets a permanent invalid-request error. Retry unchanged?", "choices": retry_choices},
    {"name": "Resource isolation", "kind": "isolate_resources",
     "question": "Slow recommendations exhaust the shared request pool. What isolates resources?", "choices": pattern_choices},
    {"name": "Client needs", "kind": "client_specific_api",
     "question": "Mobile needs small responses; web needs rich pages. What provides client-specific APIs?", "choices": pattern_choices},
    {"name": "Shared entry", "kind": "shared_entry",
     "question": "External clients need one entry point for routing and common authentication. What fits?", "choices": pattern_choices},
]
for case in cases:
    print(f"\n{case['name']}: {case['question']}")
    print("  " + " | ".join(case["choices"]))

## Step 2 · Apply explicit rules
For retries, our rule requires **both** a transient failure and an idempotent operation. Otherwise it selects B. For the other questions, the facts identify the problem: separate resource pools suggest a bulkhead, client-specific APIs suggest BFFs, and shared entry and routing suggest an API gateway.

These pattern choices address the stated need; a system can use an API gateway and BFFs together. BFFs add a layer and are not needed for every client.

In [ ]:
def rule_answer(case):
    if case["kind"] == "retry":
        if case["transient"] and case["idempotent"]:
            return "A"
        return "B"
    if case["kind"] == "isolate_resources":
        return "A"
    if case["kind"] == "client_specific_api":
        return "C"
    if case["kind"] == "shared_entry":
        return "B"
    raise ValueError("Unknown case kind")

for case in cases:
    print(case["name"], "->", rule_answer(case))

## Step 3 · Make one LLM call
Before asking all the questions, try one prompt. The prompt gives the model a short situation, three choices, and a clear instruction to answer with one letter. `ask(prompt, example=...)` sends that text to the selected model in live mode and returns its reply. There is no loop and no tool call here.

In **practice** mode, the reply is the scripted `example` value, not a model response. For a real DeepSeek response, set `MODE = "live"` and `PROVIDER = "deepseek"` in the setup cell, save `DEEPSEEK_API_KEY` in Colab Secrets, and rerun setup.

In [ ]:
prompt = """Slow recommendations exhaust the shared request pool.
Which pattern isolates resources so other services can keep working?
A. Bulkhead
B. API gateway
C. BFF
Reply with only the letter of one listed option."""

reply = ask(prompt, example="A")
print("Model reply:", reply)

## Step 3a · The same prompt with an SDK
**Why another interface?** `ask()` hides the request details so you can focus on the question. An SDK exposes the model name, the message, and the returned response. Here we use the **OpenAI Python SDK** with the OpenAI-compatible endpoint for the provider selected in setup; this does not require an OpenAI account.

Run Step 3 first so `prompt` exists. The messages include the same short system instruction used by `ask()`. In **practice** mode this cell prints a scripted answer; it does not import the SDK or call a model. In **live** mode it installs the SDK if needed, reads the selected provider's key from Colab Secrets, and makes one new API request. The request uses a timeout and no automatic retries. Provider settings still differ, and not every provider supports every SDK feature.

In [ ]:
if MODE == "practice":
    print("SDK practice reply: A (scripted; no model request)")
else:
    if find_spec("openai") is None:
        %pip install -q "openai==2.8.1"
    from openai import OpenAI
    from coursekit import PROVIDERS, secret

    settings = PROVIDERS[PROVIDER]
    options = settings.get("extra_body") or {}
    if PROVIDER == "freeinference":
        options = {"thinking": {"type": "disabled"}}
    client = OpenAI(api_key=secret(settings["secret_name"]),
                    base_url=settings["base_url"], timeout=45.0, max_retries=0)
    messages = [{"role": "system", "content": "You are a concise teaching assistant."},
                {"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=settings["model"], messages=messages, max_tokens=256, extra_body=options)
    print("SDK reply:", response.choices[0].message.content)

## Step 3b · The same prompt with LangChain
A framework provides a common interface for model calls and other building blocks. LangChain's `ChatOpenAI` adapter can call an OpenAI-compatible endpoint; `invoke(messages)` returns a message whose `content` holds the reply. We pass the same system instruction and user prompt as the SDK example. We use the **same prompt and provider** again. This is still one model call, not an agent.

Practice mode prints a scripted answer without a model call. Live mode installs the adapter if needed and sends one new request. These SDK and framework calls are **additional** to the `ask()` calls and are not counted by its 12-call limit. They can return different text or incur provider charges; review the output.

In [ ]:
if MODE == "practice":
    print("LangChain practice reply: A (scripted; no model request)")
else:
    if find_spec("langchain_openai") is None:
        %pip install -q "langchain-openai==1.1.0" "openai==2.8.1"
    from langchain_openai import ChatOpenAI
    from coursekit import PROVIDERS, secret

    settings = PROVIDERS[PROVIDER]
    options = settings.get("extra_body") or {}
    if PROVIDER == "freeinference":
        options = {"thinking": {"type": "disabled"}}
    model = ChatOpenAI(model=settings["model"], base_url=settings["base_url"],
                       api_key=secret(settings["secret_name"]), extra_body=options,
                       max_tokens=256, timeout=45, max_retries=0)
    messages = [("system", "You are a concise teaching assistant."), ("user", prompt)]
    response = model.invoke(messages)
    print("LangChain reply:", response.content)

## Step 4 · Ask an LLM the same questions
The prompt contains the question and its choices, but not the Python rules or the hidden `kind` and Boolean fields. We ask for one letter so results can be compared. The model may still be wrong or return an invalid format.

In **practice** mode, `ask` returns the scripted answers below; one is deliberately wrong. In **live** mode, it calls the selected provider once per question and ignores those scripts. This small comparison checks agreement with our simplified answer key, not general architecture expertise.

In [ ]:
practice_answers = ["A", "B", "A", "B", "A", "C", "C"]
agreement = 0
for case, practice_answer in zip(cases, practice_answers):
    options = "\n".join(case["choices"])
    prompt = f"{case['question']}\n{options}\nReply with only the letter of one listed option."
    reply = ask(prompt, example=practice_answer).strip().upper()
    expected = rule_answer(case)
    if reply == expected:
        agreement += 1
    print(f"{case['name']}: rule={expected}, model={reply}")
print(f"Agreement with the classroom rules: {agreement}/{len(cases)}")

## What you should see
Step 3, the SDK example, and the LangChain example each show a scripted **A** in practice mode. In Step 4, the rule answers are **A, B, A, B, A, C, B**. In practice mode the scripted model agrees on 6/7; the gateway answer is deliberately wrong. A live model may return different letters or extra text.

## Try it
1. Explain why a timed-out card charge without a server-supported idempotency guarantee must not be retried blindly. How could you check its status?
2. Change the product lookup's `transient` fact to `False` **and update its question** to describe a permanent error. Rerun all steps. Why does the rule change?
3. Add a new client-specific API scenario and predict the rule's answer. Then compare it with the model.
4. In live mode, compare the helper, SDK and LangChain replies. Which lines send the prompt in each example? Does a matching letter guarantee the design is correct?

## Takeaway
Rules are transparent when the relevant facts are explicit; an LLM can interpret a question written in natural language, but its answer still needs checking. Next, Lab 01 separates system and user prompts, carries context forward, and validates an answer.